# Taller — Abandono de producto financiero

**Regresión logística en Python**  
**Curso:** Machine Learning aplicado a Economía y Finanzas  
**Estudiante:** Martin Galeano

## Punto 0 — Preparación

El archivo `abandono_producto_financiero.csv` se conserva en la raíz del repositorio. Se usa una ruta relativa para que el cuaderno pueda ejecutarse desde otra copia del proyecto sin cambiar rutas locales.

## Sección 1 — Estructura de los datos

### Importación de librerías y carga de la base

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency

RUTA_DATOS = "abandono_producto_financiero.csv"
datos_crudos = pd.read_csv(RUTA_DATOS)

### Primeras observaciones

In [2]:
datos_crudos.head()

,numero_fila,id_cliente,apellido,puntaje_crediticio,pais,sexo,edad,antiguedad,saldo,numero_productos,tiene_tarjeta,miembro_activo,salario_estimado,abandono
0,1,15634602,Hargrave,619,Francia,Mujer,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,España,Mujer,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,Francia,Mujer,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,Francia,Mujer,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,España,Mujer,43,2,125510.82,1,1,1,79084.10,0


### Columnas y dimensión de la base cruda

In [3]:
print("Columnas de la base cruda:")
print(datos_crudos.columns.tolist())
print()
print("Dimensión de la base cruda:", datos_crudos.shape)

Columnas de la base cruda:
['numero_fila', 'id_cliente', 'apellido', 'puntaje_crediticio', 'pais', 'sexo', 'edad', 'antiguedad', 'saldo', 'numero_productos', 'tiene_tarjeta', 'miembro_activo', 'salario_estimado', 'abandono']

Dimensión de la base cruda: (10000, 14)


### Tipos asignados por pandas

In [4]:
datos_crudos.dtypes

numero_fila             int64
id_cliente              int64
apellido               object
puntaje_crediticio      int64
pais                   object
sexo                   object
edad                    int64
antiguedad              int64
saldo                 float64
numero_productos        int64
tiene_tarjeta           int64
miembro_activo          int64
salario_estimado      float64
abandono                int64
dtype: object

### Estadísticas descriptivas de la base cruda

In [5]:
datos_crudos.describe().T

,count,mean,std,min,25%,50%,75%,max
numero_fila,10000.0,5.000500e+03,2886.895680,1.00,2500.75,5.000500e+03,7.500250e+03,10000.00
id_cliente,10000.0,1.569094e+07,71936.186123,15565701.00,15628528.25,1.569074e+07,1.575323e+07,15815690.00
puntaje_crediticio,10000.0,6.505288e+02,96.653299,350.00,584.00,6.520000e+02,7.180000e+02,850.00
edad,10000.0,3.892180e+01,10.487806,18.00,32.00,3.700000e+01,4.400000e+01,92.00
antiguedad,10000.0,5.012800e+00,2.892174,0.00,3.00,5.000000e+00,7.000000e+00,10.00
saldo,10000.0,7.648589e+04,62397.405202,0.00,0.00,9.719854e+04,1.276442e+05,250898.09
numero_productos,10000.0,1.530200e+00,0.581654,1.00,1.00,1.000000e+00,2.000000e+00,4.00
tiene_tarjeta,10000.0,7.055000e-01,0.455840,0.00,0.00,1.000000e+00,1.000000e+00,1.00
miembro_activo,10000.0,5.151000e-01,0.499797,0.00,0.00,1.000000e+00,1.000000e+00,1.00
salario_estimado,10000.0,1.000902e+05,57510.492818,11.58,51002.11,1.001939e+05,1.493882e+05,199992.48


### Preparación del DataFrame de modelación

Según el enunciado, `numero_fila`, `id_cliente` y `apellido` son identificadores y no deben entrar en la fórmula del modelo.

In [6]:
columnas_identificadoras = ["numero_fila", "id_cliente", "apellido"]
datos = datos_crudos.drop(columns=columnas_identificadoras).copy()

# El orden fija las categorías de referencia requeridas por el profesor.
datos["pais"] = pd.Categorical(
    datos["pais"],
    categories=["Francia", "Alemania", "España"]
)
datos["sexo"] = pd.Categorical(
    datos["sexo"],
    categories=["Mujer", "Hombre"]
)

# La respuesta se conserva como variable binaria entera.
datos["abandono"] = datos["abandono"].astype("int64")

### Verificación de la estructura preparada

In [7]:
print("Dimensión para modelación:", datos.shape)
print()
print("Columnas para modelación:")
print(datos.columns.tolist())
print()
print("Tipos finales:")
print(datos.dtypes)
print()
print("Niveles de pais:", datos["pais"].cat.categories.tolist())
print("Niveles de sexo:", datos["sexo"].cat.categories.tolist())
print("Valores de abandono:", sorted(datos["abandono"].unique().tolist()))

Dimensión para modelación: (10000, 11)

Columnas para modelación:
['puntaje_crediticio', 'pais', 'sexo', 'edad', 'antiguedad', 'saldo', 'numero_productos', 'tiene_tarjeta', 'miembro_activo', 'salario_estimado', 'abandono']

Tipos finales:
puntaje_crediticio       int64
pais                  category
sexo                  category
edad                     int64
antiguedad               int64
saldo                  float64
numero_productos         int64
tiene_tarjeta            int64
miembro_activo           int64
salario_estimado       float64
abandono                 int64
dtype: object

Niveles de pais: ['Francia', 'Alemania', 'España']
Niveles de sexo: ['Mujer', 'Hombre']
Valores de abandono: [0, 1]


### Clasificación y conteo de variables

In [8]:
variables_cuantitativas = [
    "puntaje_crediticio",
    "edad",
    "antiguedad",
    "saldo",
    "numero_productos",
    "salario_estimado",
]

variables_categoricas = ["pais", "sexo"]
variables_binarias = ["tiene_tarjeta", "miembro_activo", "abandono"]

clasificacion_variables = pd.DataFrame({
    "grupo": ["Cuantitativas", "Categóricas", "Binarias"],
    "cantidad": [
        len(variables_cuantitativas),
        len(variables_categoricas),
        len(variables_binarias),
    ],
    "variables": [
        ", ".join(variables_cuantitativas),
        ", ".join(variables_categoricas),
        ", ".join(variables_binarias),
    ],
})

clasificacion_variables

,grupo,cantidad,variables
0,Cuantitativas,6,"puntaje_crediticio, edad, antiguedad, saldo, n..."
1,Categóricas,2,"pais, sexo"
2,Binarias,3,"tiene_tarjeta, miembro_activo, abandono"


### Pregunta 1. ¿Cuántas variables tiene la base cruda? ¿Cuántas quedan para el modelo?

La base cruda tiene **14 variables**. Después de retirar `numero_fila`, `id_cliente` y `apellido`, el DataFrame de modelación queda con **11 variables**: diez explicativas y la variable respuesta `abandono`.

### Pregunta 2. ¿Cuántas son cuantitativas?

De acuerdo con el diccionario del enunciado, hay **6 variables cuantitativas**: `puntaje_crediticio`, `edad`, `antiguedad`, `saldo`, `numero_productos` y `salario_estimado`. Aunque `numero_productos` toma valores enteros, el profesor la incluye entre las variables numéricas de modelación.

### Pregunta 3. ¿Cuántas son categóricas o binarias, incluyendo abandono?

Hay **2 variables categóricas** (`pais` y `sexo`) y **3 variables binarias** (`tiene_tarjeta`, `miembro_activo` y `abandono`). En conjunto son **5 variables categóricas o binarias**, incluyendo la respuesta.

### Pregunta 4. ¿Qué implica esta mezcla para el logit en Python?

Las variables cuantitativas y las binarias 0/1 pueden entrar directamente en la fórmula. En cambio, `pais` y `sexo` representan grupos sin una distancia numérica entre sus niveles. Al escribir `C(pais)` y `C(sexo)` en una fórmula de `statsmodels`, Python crea internamente los indicadores necesarios y deja una categoría como referencia, evitando incluir todas las dummies al mismo tiempo. Como los niveles fueron ordenados según el enunciado, las referencias serán **Francia** para país y **Mujer** para sexo. Esto permite interpretar los coeficientes de las demás categorías como comparaciones frente a esas referencias, sin construir las dummies manualmente.